# Density maps

## Data loading

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch

import os
from PIL import Image

BATCH_SIZE = 32
SEED = 666

# Load dataset
x_train_folder = "./data/sorghum_images/train"
x_test_folder = "./data/sorghum_images/test"
y_train_folder = "./data/sorghum_density_maps/train"
y_test_folder = "./data/sorghum_density_maps/test"

# Get a list of file names from both train and test directories
x_train_files = [os.path.splitext(file)[0] for file in os.listdir(x_train_folder) if file.endswith(".png")]
y_train_files = [os.path.splitext(file)[0] for file in os.listdir(y_train_folder) if file.endswith(".png")]
x_test_files = [os.path.splitext(file)[0] for file in os.listdir(x_test_folder) if file.endswith(".png")]
y_test_files = [os.path.splitext(file)[0] for file in os.listdir(y_test_folder) if file.endswith(".png")]

# Find matching pairs based on basenames
train_files = list(set(x_train_files).intersection(y_train_files))
test_files = list(set(x_test_files).intersection(y_test_files))

print(f"Train size: {len(train_files)}, Test size: {len(test_files)}")

# Separate transforms for input and output
input_transform = transforms.Compose([
    transforms.ToTensor(),
])

output_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Modify the custom dataset class
class SorghumDataset(torch.utils.data.Dataset):
    def __init__(self, x_folder, y_folder, file_list, input_transform=None, output_transform=None):
        self.x_folder = x_folder
        self.y_folder = y_folder
        self.file_list = file_list
        self.input_transform = input_transform
        self.output_transform = output_transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        x_path = os.path.join(self.x_folder, self.file_list[idx] + '.png')
        y_path = os.path.join(self.y_folder, self.file_list[idx] + '.png')
        
        x_image = Image.open(x_path).convert('RGB')
        y_image = Image.open(y_path).convert('L')
        
        if self.input_transform:
            x_image = self.input_transform(x_image)
        if self.output_transform:
            y_image = self.output_transform(y_image)
        
        return x_image, y_image

# Create datasets
train_dataset = SorghumDataset(x_train_folder, y_train_folder, train_files, 
                                input_transform=input_transform, 
                                output_transform=output_transform)
val_dataset = SorghumDataset(x_test_folder, y_test_folder, test_files, 
                              input_transform=input_transform, 
                              output_transform=output_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
for inputs, outputs in train_loader:
    print("Input shape:", inputs.shape)
    print("Output shape:", outputs.shape)
    break

## Training

### Loading Models

In [ ]:
import torch
from torchsummary import summary
from modelsImplementation import ASD, DeepCorn, MCNN, SACNN, SCAR, SorghumNet, SPN, CerealNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelName = "CerealNet_VGG16"

input_shape=(3, 224, 224)

#model = SCAR()
#model = SACNN()
#model = DeepCorn()
#model = SPN()
#model = ASD()
#model = MCNN.MCNN()
#model = SorghumNet.SorghumNet()
model = CerealNet.CerealNet("VGG16")

model.to(device)

summary(model, input_shape)

In [ ]:
# Example input tensor
x = torch.randn(1, 3, 224, 224).to(device)  # Batch of 1 RGB image of size 224x224

model.eval()
# Forward pass
output = model(x).to(device)
print(output.shape)

### Setting parameters

In [ ]:
import torch.optim as optim
import torch.nn as nn
import numpy as np
import os

# Loss function & optimizer
loss_fn = nn.MSELoss(reduction="sum")
optimizer = optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

early_stopping_patience = 20
best_val_loss = np.inf
best_val_mse = np.inf
patience_counter = 0

checkpoint_path = os.path.join("models", modelName + ".pt")
print(checkpoint_path)

### Training

In [ ]:
from tqdm import tqdm
from sklearn.metrics import mean_squared_error

EPOCHS = 250

old_lr = 0
new_lr = 1

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    total_mse = 0.0
    if new_lr < old_lr:
        print(f"🔻 Learning rate reduced to {new_lr:.6f}")
    # Use tqdm to wrap the DataLoader for progress bar
    with tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch") as tepoch:
        for inputs, outputs in tepoch:
            # Move tensors to GPU if available
            inputs, outputs = inputs.to(device), outputs.to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            output = model(inputs)

            # Reshape outputs to match the output shape
            loss = loss_fn(output, outputs)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Compute Mean Squared Error

            gt_array = []
            pred_array = []
            
            # Convert tensors to images
            pred_images = [transforms.ToPILImage()(pred.cpu().squeeze()) for pred in output]
            gt_images = [transforms.ToPILImage()(gt.cpu().squeeze()) for gt in outputs]

            # Convert images to NumPy arrays
            pred_arrays = [np.array(img, dtype=np.float32) for img in pred_images]
            gt_arrays = [np.array(img, dtype=np.float32) for img in gt_images]

            for pred, gt in zip(pred_arrays, gt_arrays):
                predicted = np.sum(pred) / 10000
                ground_truth = np.sum(gt) / 10000
                
                gt_array.append(ground_truth)
                pred_array.append(predicted)

            # Compute MSE between images
            mse = mean_squared_error(gt_array, pred_array) 

            # Update running loss
            running_loss += loss.item()
            total_mse += mse

            # Update progress bar description
            tepoch.set_postfix(
                loss=running_loss / (tepoch.n + 1),
                mse=total_mse / (tepoch.n + 1)
            )
    
    epoch_loss = running_loss / len(train_loader)
    epoch_MSE = total_mse / len(train_loader)

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")
    print(f"Epoch {epoch+1} MSE: {epoch_MSE:.4f}")
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_mse = 0.0

    with tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Validation", unit="batch") as vepoch:
        with torch.no_grad():  # Disable gradient calculation for validation
            for inputs, outputs in vepoch:
                # Move tensors to the device (GPU or CPU)
                inputs, outputs = inputs.to(device), outputs.to(device)

                # Forward pass
                output = model(inputs)

                loss = loss_fn(output, outputs)
                
                # Compute Mean Squared Error
                gt_array = []
                pred_array = []
                
                # Convert tensors to images
                pred_images = [transforms.ToPILImage()(pred.cpu().squeeze()) for pred in output]
                gt_images = [transforms.ToPILImage()(gt.cpu().squeeze()) for gt in outputs]

                # Convert images to NumPy arrays
                pred_arrays = [np.array(img, dtype=np.float32) for img in pred_images]
                gt_arrays = [np.array(img, dtype=np.float32) for img in gt_images]

                for pred, gt in zip(pred_arrays, gt_arrays):
                    predicted = np.sum(pred) / 10000
                    ground_truth = np.sum(gt) / 10000
                    
                    gt_array.append(ground_truth)
                    pred_array.append(predicted)

                # Compute MSE between images
                mse = mean_squared_error(gt_array, pred_array) 

                val_loss += loss.item()
                val_mse += mse

    val_loss /= len(val_loader)
    val_mse /= len(val_loader)

    print(f"Epoch {epoch+1} - Validation Loss: {val_loss:.4f}")
    print(f"Epoch {epoch+1} - Validation MSE: {val_mse:.4f}")

    old_lr = optimizer.param_groups[0]["lr"]
    lr_scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]["lr"]

    # Save best model
    if val_loss < best_val_loss and val_mse < best_val_mse:
        best_val_loss = val_loss
        best_val_mse = val_mse
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
        print(f"✅ Model improved, saved to {checkpoint_path}")
    else:
        patience_counter += 1
        print(f"🔴 No improvement, patience counter: {patience_counter}")

    # Early stopping check
    if patience_counter >= early_stopping_patience:
        print("🛑 Early stopping triggered!")
        break

## Evaluating

### Making Inferences

In [ ]:
import torch
from modelsImplementation import ASD, DeepCorn, MCNN, SACNN, SCAR, SorghumNet, SPN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelName = "CerealNet_VGG16"

input_shape=(3, 224, 224)

#model = SCAR()
#model = SACNN()
#model = DeepCorn()
#model = SPN.SPN()
#model = ASD.ASD()
#model = MCNN.MCNN()
#model = SorghumNet.SorghumNet()
model = CerealNet("VGG16")

model.to(device)
model.load_state_dict(torch.load(f'models/{modelName}.pt'))

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

input_transform = transforms.Compose([
    transforms.ToTensor(),
])

""" image_path = "./data/sorghum_images/test/112042.png"
density_map_path = "./data/sorghum_density_maps/test/112042.png" """

""" image_path = "data/sorghum_images/test/1_9_1.png"
density_map_path = "./data/sorghum_density_maps/test/1_9_1.png" """

image_path = "data/sorghum_images/test/110734.png"
density_map_path = "./data/sorghum_density_maps/test/110734.png"

image = Image.open(image_path).convert('RGB')

# GT density map
density_map = Image.open(density_map_path).convert('L')

overall_count = np.sum(density_map) / 10000

x_image = input_transform(image).to(device).unsqueeze(0)

model.eval()
with torch.no_grad():
    y_pred = model(x_image)

probability_map = transforms.ToPILImage()(y_pred.cpu().squeeze())
prediction = np.array(probability_map, dtype=np.float32)
prediction = np.sum(prediction) / 10000

# Create a figure and a 1x3 grid of subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Display the original image with title
axes[0].imshow(image)
axes[0].set_title('Original Image')

# Display the ground truth image with title
axes[1].imshow(density_map, cmap='viridis')
axes[1].set_title('Ground truth:{0}'.format(round(overall_count)))

# Display the predicted image with title
axes[2].imshow(probability_map, cmap='viridis')
axes[2].set_title('Predicted:{0}'.format(round(prediction,2)))

# Adjust layout for better spacing
plt.tight_layout()

# Show the plot
plt.show()

### Evaluating the model

In [ ]:
import pandas as pd

# Evaluation function
def data_to_evaluate(model, val_loader, device):
    model.eval()
    gt_array = []
    pred_array = []
    
    with torch.no_grad():
        for inputs, outputs in val_loader:
            # Move inputs and outputs to device
            inputs, outputs = inputs.to(device), outputs.to(device)
            
            # Get model predictions
            predictions = model(inputs)

            # Convert predictions and ground truths to images
            pred_images = [transforms.ToPILImage()(pred.cpu().squeeze()) for pred in predictions]
            gt_images = [transforms.ToPILImage()(gt.cpu().squeeze()) for gt in outputs]

            # Convert images to NumPy arrays
            pred_arrays = [np.array(img, dtype=np.float32) for img in pred_images]
            gt_arrays = [np.array(img, dtype=np.float32) for img in gt_images]

            # Compute summed values
            for pred, gt in zip(pred_arrays, gt_arrays):
                predicted = np.sum(pred) / 10000
                ground_truth = np.sum(gt) / 10000
                
                #print(f'Ground Truth: {ground_truth}, Predicted: {predicted}')
                gt_array.append(ground_truth)
                pred_array.append(predicted)

    dataset = pd.DataFrame({'Predicted': pred_array, 'Observed': gt_array})

    return dataset

# Usage
results = data_to_evaluate(model, val_loader, device)

print(results)

results.to_csv(f"results/{modelName}.csv", sep='\t')